# 06 Dodatkowe analizy – tematy (BERTopic, eksploracyjnie)

## 1. Konfiguracja

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
%cd /content/drive/MyDrive/Licencjat/extra
!pip install -q bertopic

import sys
sys.path.append("..")

import pandas as pd
from bertopic import BERTopic
from umap import UMAP

from src.config import POLITICAL_SUBREDDITS, PROCESSED_DIR, RANDOM_STATE, RESULTS_DIR

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SUBSAMPLE_PER_CAMP = 20_000

## 2. Podpróba subredditów politycznych

In [ ]:
sample = pd.read_parquet(PROCESSED_DIR / "sample.parquet", columns=["id", "subreddit", "camp", "cycle", "phase", "text"])
sample = sample[sample["subreddit"].isin(POLITICAL_SUBREDDITS)]
subsample = sample.groupby("camp").sample(SUBSAMPLE_PER_CAMP, random_state=RANDOM_STATE).reset_index(drop=True)
subsample.groupby(["camp", "cycle"]).size().unstack("cycle")

## 3. Model tematów

In [ ]:
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=RANDOM_STATE)
topic_model = BERTopic(language="english", umap_model=umap_model)
subsample["topic"], _ = topic_model.fit_transform(subsample["text"].tolist())
topic_model.get_topic_info().head(15)

## 4. Tematy obu obozów

In [ ]:
share = pd.crosstab(subsample["topic"], subsample["camp"], normalize="columns")
share = share.join(topic_model.get_topic_info().set_index("Topic")[["Count", "Name"]])
share.sort_values("Count", ascending=False).to_csv(RESULTS_DIR / "topics_camp_share.csv")
share.sort_values("Count", ascending=False).head(15).round(3)

In [ ]:
by_cycle = pd.crosstab([subsample["cycle"], subsample["camp"]], subsample["topic"], normalize="index")
by_cycle.to_csv(RESULTS_DIR / "topics_by_cycle.csv")